In [2]:
import cv2
import mediapipe as mp

# MediaPipe Face Detection সেটআপ
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# ভিডিও ক্যাপচার
cap = cv2.VideoCapture(0)  # 0 = Webcam, CCTV হলে ভিডিও ফিড URL দিবেন

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # BGR to RGB
        image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        image.flags.writeable = False

        # ফেস ডিটেকশন
        results = face_detection.process(image)

        # RGB to BGR
        image.flags.writeable = True
        image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)

        # ডিটেকশন ড্র
        if results.detections:
            for detection in results.detections:
                mp_drawing.draw_detection(image, detection)

        # ভিডিও শো
        cv2.imshow('Face Detection', image)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC চাপলে বের হবে
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1761489785.458259   60654 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1761489785.459734   60731 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) Iris(R) Xe Graphics (TGL GT2)
W0000 00:00:1761489785.463355   60725 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [3]:
import cv2

cap = cv2.VideoCapture(0)
ret, frame = cap.read()

# প্রথম ফ্রেমে ফেস ডিটেকশন (OpenCV Haarcascade)
face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
faces = face_cascade.detectMultiScale(gray, 1.3, 5)

# যদি ফেস পাওয়া যায়, প্রথম ফেস নেয়া
if len(faces) > 0:
    (x, y, w, h) = faces[0]
    tracker = cv2.TrackerKCF_create()
    tracker.init(frame, (x, y, w, h))

while True:
    ret, frame = cap.read()
    if not ret:
        break

    # ট্র্যাকার আপডেট
    success, box = tracker.update(frame)
    if success:
        x, y, w, h = [int(v) for v in box]
        cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
    else:
        cv2.putText(frame, "Lost", (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

    cv2.imshow("Face Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

In [7]:
import cv2
import dlib

detector = dlib.get_frontal_face_detector()
tracker = dlib.correlation_tracker()

cap = cv2.VideoCapture(0)
tracking = False

while True:
    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    if not tracking:
        faces = detector(gray)
        if len(faces) > 0:
            tracker.start_track(frame, faces[0])
            tracking = True
    else:
        tracker.update(frame)
        pos = tracker.get_position()
        x1, y1 = int(pos.left()), int(pos.top())
        x2, y2 = int(pos.right()), int(pos.bottom())
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)

    cv2.imshow("dlib Tracking", frame)
    if cv2.waitKey(1) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()

[ WARN:0@325.494] global cap_v4l.cpp:913 open VIDEOIO(V4L2:/dev/video0): can't open camera by index
[ERROR:0@325.494] global obsensor_uvc_stream_channel.cpp:158 getStreamChannelGroup Camera index out of range


In [5]:
import cv2
import mediapipe as mp

# MediaPipe Face Detection
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

# ভিডিও ক্যাপচার
cap = cv2.VideoCapture(0)

# ট্র্যাকার লিস্ট (একাধিক ফেস)
trackers = []
track_ids = []
frame_count = 0
DETECTION_INTERVAL = 10  # প্রতি 10 ফ্রেমে ডিটেকশন

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame_count += 1
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

        # --- ডিটেকশন --- 
        if frame_count % DETECTION_INTERVAL == 0:
            results = face_detection.process(rgb_frame)
            trackers = []
            track_ids = []

            if results.detections:
                for idx, detection in enumerate(results.detections):
                    # বাউন্ডিং বক্স
                    bboxC = detection.location_data.relative_bounding_box
                    ih, iw, _ = frame.shape
                    x, y, w, h = int(bboxC.xmin*iw), int(bboxC.ymin*ih), int(bboxC.width*iw), int(bboxC.height*ih)
                    
                    # OpenCV KCF Tracker
                    tracker = cv2.TrackerKCF_create()
                    tracker.init(frame, (x, y, w, h))
                    trackers.append(tracker)
                    track_ids.append(idx)

        # --- ট্র্যাকিং ---
        for i, tracker in enumerate(trackers):
            success, box = tracker.update(frame)
            if success:
                x, y, w, h = [int(v) for v in box]
                cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(frame, f"ID {track_ids[i]}", (x, y-10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
            else:
                cv2.putText(frame, "Lost", (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        cv2.imshow("MediaPipe + KCF Tracking", frame)
        if cv2.waitKey(1) & 0xFF == 27:
            break

cap.release()
cv2.destroyAllWindows()

I0000 00:00:1761489962.613528   60654 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1761489962.616568   60779 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) Iris(R) Xe Graphics (TGL GT2)
W0000 00:00:1761489962.625096   60773 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [6]:
import cv2
import mediapipe as mp
import numpy as np
import threading
import time

# -------------------------------
# Threaded Video Capture
# -------------------------------
class VideoStream:
    def __init__(self, src=0):
        self.cap = cv2.VideoCapture(src)
        self.ret, self.frame = self.cap.read()
        self.stopped = False

    def start(self):
        threading.Thread(target=self.update, daemon=True).start()
        return self

    def update(self):
        while not self.stopped:
            self.ret, self.frame = self.cap.read()

    def read(self):
        return self.frame

    def stop(self):
        self.stopped = True

# -------------------------------
# MediaPipe Face Detection
# -------------------------------
mp_face_detection = mp.solutions.face_detection
mp_drawing = mp.solutions.drawing_utils

DETECTION_INTERVAL = 10  # Detection every N frames

# -------------------------------
# Initialize
# -------------------------------
vs = VideoStream(0).start()
time.sleep(1.0)  # Camera warmup

trackers = []  # List of OpenCV trackers
track_ids = [] # ID for each tracker
frame_count = 0

with mp_face_detection.FaceDetection(model_selection=0, min_detection_confidence=0.5) as face_detection:
    while True:
        frame = vs.read()
        if frame is None:
            break

        frame_count += 1

        # Resize for faster processing
        small_frame = cv2.resize(frame, (640, 480))
        rgb_frame = cv2.cvtColor(small_frame, cv2.COLOR_BGR2RGB)

        # -------------------------------
        # Detection every N frames
        # -------------------------------
        if frame_count % DETECTION_INTERVAL == 0:
            results = face_detection.process(rgb_frame)
            trackers = []
            track_ids = []

            if results.detections:
                for idx, detection in enumerate(results.detections):
                    bboxC = detection.location_data.relative_bounding_box
                    ih, iw, _ = small_frame.shape
                    x, y, w, h = int(bboxC.xmin*iw), int(bboxC.ymin*ih), int(bboxC.width*iw), int(bboxC.height*ih)

                    # Initialize OpenCV KCF Tracker
                    tracker = cv2.TrackerKCF_create()
                    tracker.init(small_frame, (x, y, w, h))
                    trackers.append(tracker)
                    track_ids.append(idx)

        # -------------------------------
        # Tracking
        # -------------------------------
        for i, tracker in enumerate(trackers):
            success, box = tracker.update(small_frame)
            if success:
                x, y, w, h = [int(v) for v in box]
                cv2.rectangle(small_frame, (x, y), (x+w, y+h), (0, 255, 0), 2)
                cv2.putText(small_frame, f"ID {track_ids[i]}", (x, y-10),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
            else:
                cv2.putText(small_frame, "Lost", (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1, (0,0,255), 2)

        # Show FPS
        fps = int(vs.cap.get(cv2.CAP_PROP_FPS))
        cv2.putText(small_frame, f"FPS: {fps}", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, (255, 0, 0), 2)

        cv2.imshow("Optimized Real-Time Face Tracking", small_frame)

        if cv2.waitKey(1) & 0xFF == 27:  # ESC key
            break

vs.stop()
cv2.destroyAllWindows()

I0000 00:00:1761490053.946974   60654 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1761490053.950250   60853 gl_context.cc:369] GL version: 3.2 (OpenGL ES 3.2 Mesa 25.0.7-0ubuntu0.24.04.2), renderer: Mesa Intel(R) Iris(R) Xe Graphics (TGL GT2)
W0000 00:00:1761490053.954489   60846 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
